- In this demo we will create Customers_Silver table.
- The data in the customers topic contains complete row output from a Change Data Capture Feed
- The changes captured are either INSERT, UPDATE or DELETE

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
from pyspark.sql import functions as F

schema = "customer_id STRING, email STRING, first_name STRING, last_name STRING, gender STRING, street STRING, city STRING, country_code STRING, row_status STRING, row_time TIMESTAMP"

customers_df = (
                spark.table("bronze")
                .filter("topic = 'customers'")
                .select(F.from_json(F.col("value").cast("string"),schema).alias("v"))
                .select("v.*")
                .filter("row_status in ('insert', 'update')")
)

display(customers_df)

In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())

ranked_df = (customers_df.withColumn("rank",F.rank().over(window))
                .filter("rank = 1")
                .drop("rank")
)

display(ranked_df)

Now Apply Same logic to Streaming Read

In [0]:
ranked_df = (
                spark.readStream.table("bronze")
                .filter("topic = 'customers'")
                .select(F.from_json(F.col("value").cast("string"),schema).alias("v"))
                .select("v.*")
                .filter("row_status in ('insert','update')")
                .withColumn("rank", F.rank().over(window))
                .filter("rank == 1")
                .drop("rank")

)
(ranked_df.writeStream
            .option("checkpointLocation", f"{bookstore.checkpoint_path}/ranked")
            .trigger(availableNow=True)
            .format("console")
            .start()
)

#display(ranked_df, checkpointLocation = f"{bookstore.checkpoint_path}/customers_silver_temp")

In [0]:
from pyspark.sql.window import Window

def batch_upsert(microBatchDF,batchId):
    window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())

    (microBatchDF.filter("row_status in ('insert','update')")
        .withColumn("rank", F.rank().over(window))
        .filter("rank == 1")
        .drop("rank")
        .createOrReplaceTempView("ranked_updates")
     )


    query = """
        MERGE INTO customers_silver c
        USING ranked_updates r
        ON c.customer_id=r.customer_id
            WHEN MATCHED AND c.row_time < r.row_time
              THEN UPDATE SET *
            WHEN NOT MATCHED
              THEN INSERT *
    """

    microBatchDF.sparkSession.sql(query)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customers_silver
(customer_id STRING, email STRING, first_name STRING, last_name STRING, gender STRING, street STRING, city STRING, country STRING, row_time TIMESTAMP)

In [0]:
df_country_lookup = spark.read.json(f"{bookstore.dataset_path}/country_lookup")
display(df_country_lookup)

In [0]:
query = (
        spark.readStream
                .table("bronze")
                .filter("topic = 'customers'")
                .select(F.from_json(F.col("value").cast("string"),schema).alias("v"))
                .select("v.*")
                .join(F.broadcast(df_country_lookup), F.col("country_code") == F.col("code") , "inner")
            .writeStream
                .foreachBatch(batch_upsert)
                .option("checkpointLocation", f"{bookstore.checkpoint_path}/customers_silver")
                .trigger(availableNow=True)
                .start()
)

query.awaitTermination()

In [0]:
count = spark.table("customers_silver").count()
expected_count = spark.table("customers_silver").select("customer_id").distinct().count()

assert count == expected_count
print("Unit Test Passed")